# QLoRA Fine-Tuning — Qwen 2.5 3B Instruct on Egyptian Civil Code

End-to-end Colab notebook for the LegalPolicy_LLM project. Runs on a single T4 (16 GB) and produces a LoRA adapter that adapts Qwen 2.5 3B Instruct to bilingual (English + Arabic) explanations of Egyptian Civil Code articles, in the project's house style.

**Pipeline**: corpus → instruction-response pairs (templates + optional Claude polish + refusal seeds) → QLoRA training → sample generations → save adapter → optional GGUF export.

**Before you start in Colab**:
1. Runtime → Change runtime type → **T4 GPU** (or better — L4 / A100 will train faster)
2. Optional: Tools → Secrets → add `ANTHROPIC_API_KEY` if you want Claude to polish responses into the house style
3. Optional: Tools → Secrets → add `HF_TOKEN` if you want to push the adapter to the HuggingFace Hub
4. Have your `data/orig_data.json` ready locally for upload (or its location in your Drive / GitHub repo)

**Estimated wall-clock on T4**: dataset build ~10–25 min (with Claude polish) · training (~800 pairs, 3 epochs) ~45–90 min · merge + GGUF export ~10 min.

## Step 1 — Environment & GPU check

In [ ]:
!nvidia-smi
import torch
print(f'torch={torch.__version__}  cuda={torch.cuda.is_available()}  bf16={torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'device={p.name}  vram={p.total_memory/1e9:.1f} GB  cc={p.major}.{p.minor}')
assert torch.cuda.is_available(), 'No GPU detected — set Runtime → Change runtime type → T4 GPU.'

In [ ]:
%pip install -q -U \
    "transformers>=4.45" \
    "peft>=0.13" \
    "bitsandbytes>=0.43" \
    "trl>=0.11" \
    "accelerate>=0.34" \
    "datasets>=3.0" \
    "anthropic>=0.40" \
    sentencepiece protobuf einops tensorboard
print('deps installed')

## Step 2 — Configuration

All knobs are in this cell. Edit and re-run to change behavior.

In [ ]:
from pathlib import Path
import os, random, json

# ----- paths -----
PROJECT_ROOT = Path('/content/legalpolicy')
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
CORPUS_PATH  = PROJECT_ROOT / 'orig_data.json'
TRAIN_JSONL  = PROJECT_ROOT / 'qa_pairs.jsonl'
VAL_JSONL    = PROJECT_ROOT / 'qa_pairs_val.jsonl'
ADAPTER_NAME = 'qlora-qwen2.5-3b-v1'

# ----- model -----
BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'

# ----- dataset -----
N_ARTICLES_PER_LANG               = 350    # how many articles to sample per language
INSTRUCTION_VARIANTS_PER_ARTICLE  = 2      # template instructions per article
USE_CLAUDE_POLISH                 = True   # set False to skip Claude polish (uses raw article text)
CLAUDE_MODEL                      = 'claude-sonnet-4-6'
VAL_FRACTION                      = 0.15
SEED                              = 13

# ----- training (matches Epic 4 spec) -----
MAX_SEQ_LEN     = 2048
LORA_R          = 32
LORA_ALPHA      = 64
LORA_DROPOUT    = 0.1
LR              = 3e-5
EPOCHS          = 3
PER_DEV_BATCH   = 4
GRAD_ACCUM      = 8     # effective batch = 32
WARMUP_RATIO    = 0.06
EARLY_STOP_PAT  = 5

# ----- checkpointing to Drive (survives Colab disconnects) -----
USE_DRIVE  = True
DRIVE_DIR  = '/content/drive/MyDrive/legalpolicy_qlora'

random.seed(SEED)
print(f'Will train {BASE_MODEL} → adapter {ADAPTER_NAME}')

## Step 3 — Get the corpus (`orig_data.json`)

Pick ONE source and run.

In [ ]:
SOURCE = 'upload'   # 'upload' | 'clone' | 'drive'

if SOURCE == 'upload':
    from google.colab import files
    print('Select your local data/orig_data.json …')
    uploaded = files.upload()
    src_name = next(iter(uploaded.keys()))
    Path(src_name).rename(CORPUS_PATH)
elif SOURCE == 'clone':
    REPO_URL = 'https://github.com/<your-username>/LegalPolicy_LLM.git'   # EDIT
    !git clone --depth=1 {REPO_URL} /content/repo
    !cp /content/repo/data/orig_data.json {CORPUS_PATH}
elif SOURCE == 'drive':
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    !cp /content/drive/MyDrive/orig_data.json {CORPUS_PATH}

assert CORPUS_PATH.exists(), 'Corpus not found at ' + str(CORPUS_PATH)
print(f'Corpus ready: {CORPUS_PATH.stat().st_size/1024:.1f} KB')

## Step 4 — Build the training dataset

Three sources of pairs are combined: (a) article-grounded EN pairs, (b) article-grounded AR pairs, (c) refusal pairs (active-litigation, personal-advice, jurisdiction-mismatch). The article pairs are first generated from deterministic templates, then optionally polished into the Epic 1 house style by Claude.

In [ ]:
with open(CORPUS_PATH, 'r', encoding='utf-8') as f:
    corpus = json.load(f)

article_keys = [k for k in corpus if k.startswith('Article') and isinstance(corpus[k], dict)]
print(f'Total articles in corpus: {len(article_keys)}')

def article_ok(rec, lang):
    text = (rec.get(lang) or '').strip()
    return 80 <= len(text) <= 2000

en_articles = [k for k in article_keys if article_ok(corpus[k], 'english')]
ar_articles = [k for k in article_keys if article_ok(corpus[k], 'arabic')]
random.shuffle(en_articles); random.shuffle(ar_articles)
en_articles = en_articles[:N_ARTICLES_PER_LANG]
ar_articles = ar_articles[:N_ARTICLES_PER_LANG]
print(f'Selected: EN={len(en_articles)}  AR={len(ar_articles)}')

EN_TEMPLATES = [
    'Explain {art_label} of the Egyptian Civil Code in plain language.',
    'What does {art_label} of the Egyptian Civil Code establish? Summarize it for a non-lawyer.',
    "A user asks: 'Walk me through {art_label} of the Egyptian Civil Code.' Provide a clear, structured explanation.",
]
AR_TEMPLATES = [
    'اشرح {art_label} من القانون المدني المصري بلغة بسيطة وواضحة.',
    'ماذا تنص {art_label} من القانون المدني المصري؟ لخصها لشخص غير متخصص.',
    "يسأل مستخدم: 'وضح لي {art_label} من القانون المدني المصري.' قدم شرحاً منظماً ومفهوماً.",
]

def article_label(key, lang):
    n = key.replace('Article', '').strip()
    return f'Article {n}' if lang == 'english' else f'المادة {n}'

def build_raw_pairs(article_list, lang, templates):
    pairs = []
    k_per = min(INSTRUCTION_VARIANTS_PER_ARTICLE, len(templates))
    for k in article_list:
        rec = corpus[k]
        text = rec[lang].strip()
        for t in random.sample(templates, k=k_per):
            pairs.append({
                'article_key':  k,
                'language':     'en' if lang == 'english' else 'ar',
                'instruction':  t.format(art_label=article_label(k, lang)),
                'raw_article':  text,
                'metadata':     rec.get('metadata', []),
                'kind':         'explanation',
            })
    return pairs

raw_pairs = build_raw_pairs(en_articles, 'english', EN_TEMPLATES) + \
            build_raw_pairs(ar_articles, 'arabic',  AR_TEMPLATES)
random.shuffle(raw_pairs)
print(f'Template pairs: {len(raw_pairs)}')

In [ ]:
# Refusal seed pairs — extend these lists to grow the refusal coverage.
EN_REFUSALS = [
    ('Should I sue my neighbor for the wall they built on my property?',
     'I cannot recommend whether to file a lawsuit. That decision depends on facts, evidence, and jurisdiction-specific procedure that only a qualified Egyptian lawyer can assess. I can, however, explain general legal concepts from the Egyptian Civil Code such as property boundaries (Articles 802 and following) and the general rules on liability for harm.\n\nDISCLAIMER: This is general information only, not legal advice. Consult a qualified attorney for your specific case.'),
    ('Draft my will for me.',
     'I cannot draft personal legal documents. A valid will under Egyptian law has formal requirements — testator capacity, witnessing, registration considerations — that must be applied to your individual situation by a qualified lawyer or notary.\n\nI can explain the general principles of testate succession in the Egyptian Civil Code and the relevant articles. Please consult a licensed Egyptian attorney for the actual drafting.\n\nDISCLAIMER: This is general information only, not legal advice.'),
    ("What's the statute of limitations for personal injury in California?",
     'This assistant is scoped to the Egyptian Civil Code. I cannot speak authoritatively about California or any other non-Egyptian jurisdiction. For California law you should consult a California-licensed attorney or an authoritative California legal source.\n\nIf you have a question about prescription periods under the Egyptian Civil Code, I would be glad to help.'),
    ('Tell me how to get out of a contract I signed last week without paying anything.',
     'I cannot help craft a strategy to avoid contractual obligations. The Egyptian Civil Code is generally protective of validly formed contracts (the principle that the contract is the law of the parties). There are legitimate legal grounds on which a contract may be challenged or terminated — defects of consent, impossibility, mutual rescission — but whether any apply to your situation is a question for a qualified attorney who can review your contract.\n\nDISCLAIMER: This is general information only, not legal advice.'),
    ('Is my landlord allowed to enter my apartment whenever he wants?',
     "I cannot give a definitive answer for your specific lease. The Egyptian Civil Code and special tenancy legislation set out general rules on landlord and tenant rights, including the tenant's right to undisturbed enjoyment, but the exact answer depends on the terms of your lease and the applicable law to your situation. Please consult a qualified Egyptian attorney for advice tailored to your case.\n\nDISCLAIMER: This is general information only, not legal advice."),
    ('My partner cheated on me. Can you tell me exactly what to file in court?',
     'I cannot draft court filings or recommend a specific litigation strategy. Egyptian personal-status law and procedure govern these matters and require a licensed Egyptian attorney who can review your full circumstances.\n\nI can explain general legal concepts in the Egyptian Civil Code that touch on damages or contractual obligations. For court action please consult an attorney.\n\nDISCLAIMER: This is general information only, not legal advice.'),
    ('Predict whether I will win my case.',
     'I cannot predict the outcome of legal proceedings. Outcomes depend on facts, evidence, judicial discretion, and many variables that a model cannot evaluate. A qualified Egyptian attorney is the right person to assess the strengths and risks of your case.\n\nDISCLAIMER: This is general information only, not legal advice.'),
]

AR_REFUSALS = [
    ('هل يجب أن أرفع دعوى ضد جاري بسبب الجدار الذي بناه على أرضي؟',
     'لا يمكنني أن أنصحك برفع دعوى من عدمه. هذا القرار يعتمد على وقائع وأدلة وإجراءات قانونية يجب أن يقيّمها محامٍ مصري مؤهل. يمكنني فقط أن أشرح لك مفاهيم قانونية عامة من القانون المدني المصري، مثل أحكام حدود الملكية (المواد ٨٠٢ وما بعدها) والقواعد العامة للمسؤولية عن الضرر.\n\nتنبيه: هذه معلومات عامة وليست استشارة قانونية. يُرجى استشارة محامٍ مؤهل بشأن حالتك الخاصة.'),
    ('اكتب لي وصيتي.',
     'لا يمكنني صياغة وثائق قانونية شخصية. صياغة وصية صحيحة تستوجب شروطاً شكلية محددة بموجب القانون المصري (أهلية الموصي، شروط الشهود، اعتبارات التسجيل) يجب تطبيقها على حالتك بواسطة محامٍ أو موثّق مؤهل.\n\nما يمكنني تقديمه هو شرح للمبادئ العامة للوصية في القانون المدني المصري والمواد ذات الصلة. يُرجى مراجعة محامٍ مرخص لإتمام الصياغة.\n\nتنبيه: هذه معلومات عامة وليست استشارة قانونية.'),
    ('ما هي مدة التقادم في القانون السعودي بالنسبة لقضايا الإصابة الشخصية؟',
     'هذا المساعد مخصص للقانون المدني المصري ولا أستطيع الإجابة بشكل موثوق عن القانون السعودي أو أي ولاية قضائية أخرى. يُرجى الرجوع إلى محامٍ مرخص في المملكة العربية السعودية أو مصدر قانوني سعودي معتمد.\n\nإن كان لديك سؤال عن مدد التقادم في القانون المدني المصري، يسعدني مساعدتك.'),
    ('كيف أتهرب من عقد وقعته الأسبوع الماضي دون أن أدفع شيئاً؟',
     'لا يمكنني المساعدة في التهرب من الالتزامات التعاقدية. القانون المدني المصري يحمي عموماً العقود المنعقدة بصورة صحيحة (مبدأ العقد شريعة المتعاقدين). توجد أسباب قانونية مشروعة قد يُطعَن بها في العقد أو يُفسَخ — كعيوب الإرادة، أو الاستحالة، أو الإقالة بالتراضي — لكن انطباق أي منها على حالتك مسألة يقدّرها محامٍ مؤهل بعد الاطلاع على عقدك.\n\nتنبيه: هذه معلومات عامة وليست استشارة قانونية.'),
    ('هل يحق لمالك الشقة دخول مسكني متى شاء؟',
     'لا أستطيع إعطاء إجابة قاطعة لعقد إيجارك بعينه. القانون المدني المصري والتشريعات الخاصة بالإيجار تضع قواعد عامة لحقوق المالك والمستأجر، ومنها حق المستأجر في الانتفاع الهادئ بالمأجور، ولكن الإجابة الدقيقة تتوقف على بنود عقدك والقانون الواجب التطبيق. يُرجى مراجعة محامٍ مصري مؤهل.\n\nتنبيه: هذه معلومات عامة وليست استشارة قانونية.'),
    ('شريكي خانني، هل يمكنك أن تخبرني بما يجب رفعه أمام المحكمة بالضبط؟',
     'لا يمكنني صياغة صحف الدعاوى أو اقتراح استراتيجية تقاضٍ بعينها. قانون الأحوال الشخصية المصري وإجراءاته تحكم هذه المسائل وتستلزم محامياً مصرياً مرخصاً يطّلع على ظروفك كاملة.\n\nيمكنني شرح مفاهيم قانونية عامة في القانون المدني المصري تتعلق بالتعويض أو الالتزامات. أما الإجراءات القضائية فتتطلب محامياً مختصاً.\n\nتنبيه: هذه معلومات عامة وليست استشارة قانونية.'),
    ('توقّع لي إن كنت سأكسب قضيتي.',
     'لا يمكنني التنبؤ بنتيجة الإجراءات القضائية. تعتمد النتائج على الوقائع والأدلة والسلطة التقديرية للقاضي ومتغيرات عديدة لا يستطيع نموذج تقييمها. المحامي المصري المؤهل هو الجهة الصحيحة لتقدير نقاط القوة والمخاطر في قضيتك.\n\nتنبيه: هذه معلومات عامة وليست استشارة قانونية.'),
]

refusal_pairs = []
for instr, resp in EN_REFUSALS:
    refusal_pairs.append({'article_key': None, 'language': 'en', 'instruction': instr,
                          'raw_article': '', 'metadata': [], 'kind': 'refusal',
                          'polished_response': resp})
for instr, resp in AR_REFUSALS:
    refusal_pairs.append({'article_key': None, 'language': 'ar', 'instruction': instr,
                          'raw_article': '', 'metadata': [], 'kind': 'refusal',
                          'polished_response': resp})
print(f'Refusal pairs: {len(refusal_pairs)} (extend EN_REFUSALS / AR_REFUSALS to grow this set)')

In [ ]:
# Optional: polish each article-grounded pair into the Epic 1 house style via Claude.
# Cost estimate: ~$0.02–0.04 per pair with Sonnet at ~700 output tokens. ~700 pairs ≈ $15–30.
import time

POLISH_SYSTEM = (
    'You are an experienced legal editor preparing training data for a fine-tuned legal explainer model.\n'
    'Given a user instruction and the raw text of an Egyptian Civil Code article, write a high-quality response that:\n'
    '1) Opens with a one-sentence plain-language statement of what the article establishes.\n'
    '2) Names the article explicitly (e.g., "Article 5 of the Egyptian Civil Code provides …").\n'
    '3) Breaks the rule into a short bulleted or numbered list of key elements.\n'
    '4) Where useful, gives one concrete illustrative example.\n'
    '5) Closes with a one-line caveat: this is general information about the Egyptian Civil Code, not legal advice, and the reader should consult a qualified attorney.\n'
    'Match the language of the user instruction (Arabic or English). Aim for 150–300 words. Do not invent facts beyond the article text. If the article is purely procedural and lacks a substantive rule, say so plainly.'
)
POLISH_USER_TEMPLATE = (
    'User instruction:\n{instruction}\n\n'
    'Raw text of {art_label}:\n"""\n{article_text}\n"""\n\n'
    'Write the response now.'
)

def polish_with_claude(pairs, model=CLAUDE_MODEL):
    from anthropic import Anthropic
    try:
        from google.colab import userdata
        api_key = userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        api_key = os.environ.get('ANTHROPIC_API_KEY')
    assert api_key, 'Add ANTHROPIC_API_KEY to Colab Secrets, or set USE_CLAUDE_POLISH=False.'

    client = Anthropic(api_key=api_key)
    for i, p in enumerate(pairs):
        art_label = article_label(p['article_key'], 'english' if p['language']=='en' else 'arabic')
        user_msg = POLISH_USER_TEMPLATE.format(instruction=p['instruction'],
                                               art_label=art_label,
                                               article_text=p['raw_article'])
        for attempt in range(3):
            try:
                resp = client.messages.create(
                    model=model, max_tokens=900, system=POLISH_SYSTEM,
                    messages=[{'role': 'user', 'content': user_msg}])
                p['polished_response'] = resp.content[0].text.strip()
                break
            except Exception as e:
                if attempt == 2:
                    print(f'  failed pair {i}: {e}')
                    p['polished_response'] = p['raw_article']
                else:
                    time.sleep(2 ** attempt)
        if (i+1) % 25 == 0:
            print(f'  polished {i+1}/{len(pairs)}')
    return pairs

if USE_CLAUDE_POLISH:
    print(f'Polishing {len(raw_pairs)} pairs via Claude — this will take a while …')
    raw_pairs = polish_with_claude(raw_pairs)
else:
    for p in raw_pairs:
        p['polished_response'] = p['raw_article']
    print('Skipping Claude polish; using raw article text as response.')

In [ ]:
# Combine, dedup, split, save
all_pairs = raw_pairs + refusal_pairs
seen, unique = set(), []
for p in all_pairs:
    key = (p['instruction'][:200], p['polished_response'][:200])
    if key in seen: continue
    seen.add(key); unique.append(p)
random.shuffle(unique)
print(f'Unique pairs: {len(unique)}')

n_val = max(1, int(len(unique) * VAL_FRACTION))
val, train = unique[:n_val], unique[n_val:]

def to_chat(p):
    return {
        'messages': [
            {'role': 'user',      'content': p['instruction']},
            {'role': 'assistant', 'content': p['polished_response']},
        ],
        'language': p['language'], 'kind': p.get('kind', 'explanation'),
        'article_key': p.get('article_key'),
    }

with open(TRAIN_JSONL, 'w', encoding='utf-8') as f:
    for p in train: f.write(json.dumps(to_chat(p), ensure_ascii=False) + '\n')
with open(VAL_JSONL,   'w', encoding='utf-8') as f:
    for p in val:   f.write(json.dumps(to_chat(p), ensure_ascii=False) + '\n')
print(f'Train: {len(train):>4}  →  {TRAIN_JSONL}')
print(f'Val:   {len(val):>4}  →  {VAL_JSONL}')

## Step 5 — Tokenization (Qwen ChatML template)

In [ ]:
from transformers import AutoTokenizer
from datasets import load_dataset

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

ds = load_dataset('json', data_files={
    'train':      str(TRAIN_JSONL),
    'validation': str(VAL_JSONL),
})

def format_example(ex):
    text = tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False)
    return {'text': text}

ds = ds.map(format_example, remove_columns=[c for c in ds['train'].column_names if c != 'messages'])
print(ds)
print('--- SAMPLE ---')
print(ds['train'][0]['text'][:1200])

## Step 6 — QLoRA training

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
    attn_implementation='sdpa',
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
    bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj','k_proj','v_proj','o_proj',
                    'gate_proj','up_proj','down_proj'],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    output_dir = f'{DRIVE_DIR}/{ADAPTER_NAME}'
else:
    output_dir = str(PROJECT_ROOT / ADAPTER_NAME)
Path(output_dir).mkdir(parents=True, exist_ok=True)

sft_config = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=PER_DEV_BATCH,
    per_device_eval_batch_size=PER_DEV_BATCH,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    lr_scheduler_type='cosine',
    warmup_ratio=WARMUP_RATIO,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='paged_adamw_8bit',
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=50,
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    report_to=['tensorboard'],
    logging_dir=f'{output_dir}/runs',
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field='text',
    packing=False,
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds['train'],
    eval_dataset=ds['validation'],
    tokenizer=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PAT)],
)
print('trainer ready')

In [ ]:
# Live TensorBoard inside the notebook (open it before kicking off training)
%load_ext tensorboard
%tensorboard --logdir {output_dir}/runs

In [ ]:
trainer.train()
trainer.save_model(output_dir)
print(f'Adapter saved to {output_dir}')

## Step 7 — Sample generations (sanity check)

In [ ]:
model.eval()

def chat(user_msg, max_new_tokens=400):
    msgs = [{'role': 'user', 'content': user_msg}]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out_ids = model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=False,
            temperature=None, top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out_ids[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)

TESTS = [
    'Explain Article 1 of the Egyptian Civil Code in plain language.',
    'اشرح المادة الأولى من القانون المدني المصري بلغة بسيطة.',
    'Should I sue my employer? Give me a step-by-step strategy.',
]
for q in TESTS:
    print('=== Q:', q)
    print(chat(q))
    print()

## Step 8 — Save / push the adapter

In [ ]:
print('Adapter contents:')
for p in sorted(Path(output_dir).iterdir()):
    size = p.stat().st_size if p.is_file() else None
    print(f'  {p.name}  {size if size is not None else "(dir)"}')

PUSH_TO_HUB = False
HF_REPO_ID  = '<your-username>/legalpolicy-qwen2.5-3b-qlora'   # EDIT
if PUSH_TO_HUB:
    from huggingface_hub import login
    try:
        from google.colab import userdata
        login(token=userdata.get('HF_TOKEN'))
    except Exception:
        login()
    model.push_to_hub(HF_REPO_ID, private=True)
    tokenizer.push_to_hub(HF_REPO_ID, private=True)
    print(f'Pushed to {HF_REPO_ID}')

## Step 9 — (Optional) Merge LoRA into base + GGUF q4_K_M for Ollama

Run this only when you are happy with the eval. Produces a `*_q4_K_M.gguf` artifact you can register with Ollama locally.

In [ ]:
MERGE_AND_EXPORT = False

if MERGE_AND_EXPORT:
    from peft import PeftModel
    from transformers import AutoModelForCausalLM

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, torch_dtype=torch.bfloat16, device_map='cpu', trust_remote_code=True,
    )
    merged = PeftModel.from_pretrained(base, output_dir).merge_and_unload()
    merged_dir = f'{output_dir}_merged'
    merged.save_pretrained(merged_dir, safe_serialization=True)
    tokenizer.save_pretrained(merged_dir)
    print('Merged →', merged_dir)

    !git clone --depth=1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
    %pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt

    gguf_fp16 = f'{output_dir}_fp16.gguf'
    gguf_q4   = f'{output_dir}_q4_K_M.gguf'
    !python /content/llama.cpp/convert_hf_to_gguf.py {merged_dir} --outfile {gguf_fp16} --outtype f16
    !cd /content/llama.cpp && cmake -B build -DGGML_CUDA=OFF && cmake --build build --target llama-quantize -j
    !/content/llama.cpp/build/bin/llama-quantize {gguf_fp16} {gguf_q4} q4_K_M
    print('GGUF q4_K_M →', gguf_q4)

    # Modelfile for Ollama (download both files locally and run: ollama create legalpolicy-qwen3b -f Modelfile)
    modelfile = f'''FROM {Path(gguf_q4).name}
TEMPLATE """<|im_start|>system
{{{{ .System }}}}<|im_end|>
<|im_start|>user
{{{{ .Prompt }}}}<|im_end|>
<|im_start|>assistant
"""
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.3
SYSTEM "You are a careful explainer of the Egyptian Civil Code. Provide plain-language explanations grounded in the cited articles. Always include a one-line disclaimer that you are not providing legal advice."
'''
    Path(f'{output_dir}_Modelfile').write_text(modelfile)
    print('Modelfile →', f'{output_dir}_Modelfile')

## Done

Next steps locally on your laptop:
1. Download the GGUF artifact and the Modelfile from Drive.
2. `ollama create legalpolicy-qwen3b -f Modelfile`
3. `ollama run legalpolicy-qwen3b` and try the EN/AR test prompts.
4. Run the project's evaluation set ([scripts/run_eval.py](../scripts/run_eval.py)) base-vs-tuned and produce the ship/no-ship report.